<a href="https://colab.research.google.com/github/Muqtarali/Generative-AI-using-LangChain/blob/main/agents-in-langchain/agents_in_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
os.environ["GROQ_API_KEY"]=''
os.environ["GOOGLE_API_KEY"]=''

In [ ]:
!pip install -U langchain-community langchain-core requests duckduckgo-search  langchain-groq

In [ ]:
!pip install -U ddgs duckduckgo-search

In [ ]:
from langchain_groq.chat_models import ChatGroq
from langchain_core.tools import tool
import requests

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool=DuckDuckGoSearchRun()

result=search_tool.invoke('top news in india today')

/tmp/ipykernel_6445/1720390147.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [ ]:
result

"Hindustan Times: Stay updated with Hindustan Times for the top India news, world events, and breaking stories. Get real-time updates on politics, wars, live cricket scores, entertainment news, and ... India News | Latest India News | Read latest and breaking news from India. Today's top India news headlines, news on Indian politics, elections, government, business, technology, and Bollywood. Stay updated with the latest breaking news, live updates, photos, videos and top trending stories from India and around the world across politics, economy, sports and more on The Economic Times. Live News: Check our live blog for real-time updates on trending news, breaking news from India, World, Sports, entertainment, politics and more. The Times of India Covers all latest breaking news ... Get the latest India news, breaking headlines, political developments, government updates, weather alerts, crime news and state coverage at CNBC TV18."

In [ ]:
@tool
def get_weather_data(city: str) -> str:
  """
  This function fetches the current weather data for a given city
  """
  url = f'https://api.weatherstack.com/current?access_key=4d1d8ae207a8c845a52df8a67bf3623e&query={city}'

  response = requests.get(url)

  return response.json()

In [ ]:
model = ChatGroq(
    model='openai/gpt-oss-20b',
    temperature=0.2,
)

In [ ]:
%pip install -U langchainhub

In [ ]:
from langchain.agents import create_agent
from langchain_classic.agents import AgentExecutor
from langchain_classic import hub

In [ ]:
%pip install -U langsmith

In [ ]:
import inspect
print(inspect.signature(client.pull_prompt))

(prompt_identifier: 'str', *, include_model: 'bool | None' = False, secrets: 'dict[str, str] | None' = None, secrets_from_env: 'bool' = False, skip_cache: 'bool' = False, dangerously_pull_public_prompt: 'bool' = False) -> 'Any'


In [ ]:
prompt = client.pull_prompt(
    "hwchase17/react",
    dangerously_pull_public_prompt=True,
)

In [ ]:
agent = create_agent(
    model=model,
    tools=[search_tool,get_weather_data],
    system_prompt="Give practical travel options. Say when schedules or prices need checking.",
)

In [ ]:
# Step 4: Wrap it with AgentExecutor
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool,get_weather_data],
    verbose=True
)

In [ ]:
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "Find the capital of Madhya Pradesh, then find it's current weather condition"}
    ]
})
print(result)

{'messages': [HumanMessage(content='3 ways to reach Goa from Delhi', additional_kwargs={}, response_metadata={}, id='cfd64f73-6f7b-4f85-8e41-96090ab7ceb5'), AIMessage(content='**Three practical ways to get from Delhi to Goa**\n\n| # | Mode of travel | How it works | Approx. travel time | Typical cost (₹) | When to check schedules/prices |\n|---|----------------|--------------|---------------------|------------------|--------------------------------|\n| 1 | **Flight** | Direct or one‑stop flights from Indira Gandhi International (DEL) to Dabolim Airport (GOI).  | 2\u202f–\u202f3\u202fhrs flight + 1–2\u202fhrs airport transfer | ₹3\u202f000\u202f–\u202f₹12\u202f000 (economy, varies by season & booking window) | **Always** – flight times, seat availability, and fares change daily. Check airlines (IndiGo, Air India, SpiceJet, GoAir, Vistara) or aggregators (MakeMyTrip, Cleartrip, Google Flights). |\n| 2 | **Train** | Overnight or day trains from New Delhi (NDLS) to Goa’s major stations (Ma

In [ ]:
response['output']